# KLE Single Colab Notebook

一个可直接 `Run all` 的单文件 notebook：

1. 自动安装依赖并拉取仓库
2. 读取 `data/data.csv`
3. 运行 Hybrid 模型 walk-forward 回测
4. 运行小规模网格搜索（可选）
5. 用最优配置训练最终模型并给出下一期 Top20 与推荐票

> 说明：彩票本质随机，本 notebook 仅用于研究。

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def run(cmd: str) -> None:
    print(f"+ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


REPO_URL = "https://github.com/jursmatsko/lotto-image-predictor-research.git"
REPO_ROOT = Path("/content/lotto-image-predictor-research")
PROJECT_DIR = REPO_ROOT / "kle"

run("python -m pip install -q numpy pandas matplotlib scikit-learn tqdm torch")

if not REPO_ROOT.exists():
    run(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    print(f"Repo already exists: {REPO_ROOT}")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"Python: {sys.version.split()[0]}")
print(f"Working dir: {Path.cwd()}")

In [ ]:
import pandas as pd


data_path = PROJECT_DIR / "data" / "data.csv"
df = pd.read_csv(data_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
display(df.head())
display(df.tail())

In [ ]:
# ===== 全局参数 =====
LAST_N = 20
TRAIN_WINDOW = 320

# Hybrid 融合权重
W_ML = 0.42
W_FREQ = 0.30
W_GAP = 0.18
W_MOM = 0.10

# RF / ET 参数
RF_TREES = 240
RF_DEPTH = 9
ET_TREES = 320
ET_DEPTH = 10

# RF + Transformer
USE_RF_TRANSFORMER = False
TRANS_SEQ_LEN = 30
TRANS_EPOCHS = 4
TRANS_BATCH = 64
TRANS_D_MODEL = 64
TRANS_HEADS = 4
W_TRANS = 0.22   # 在最终融合中给 Transformer 的权重

NUM_TICKETS = 10

print("Config loaded")
print(f"LAST_N={LAST_N}, TRAIN_WINDOW={TRAIN_WINDOW}")
print(f"Weights: ML={W_ML}, FREQ={W_FREQ}, GAP={W_GAP}, MOM={W_MOM}")
print(f"RF+Transformer: enabled={USE_RF_TRANSFORMER}, seq_len={TRANS_SEQ_LEN}, epochs={TRANS_EPOCHS}, w_trans={W_TRANS}")

In [ ]:
import numpy as np
from dataclasses import dataclass
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

TOTAL_NUMBERS = 80
DRAW_SIZE = 20
FEAT_WINDOWS = [5, 10, 20, 50]


def load_draws(df):
    issue_col = "期数" if "期数" in df.columns else "期号"
    number_cols = [f"红球_{i}" for i in range(1, 21)]
    x = df.sort_values(issue_col, ascending=True).reset_index(drop=True)

    issues, draws = [], []
    for _, row in x.iterrows():
        nums = [int(row[c]) for c in number_cols if 1 <= int(row[c]) <= 80]
        if len(nums) == 20:
            issues.append(str(row[issue_col]))
            draws.append(nums)
    return issues, draws


def presence_matrix(draws):
    p = np.zeros((len(draws), TOTAL_NUMBERS), dtype=np.float32)
    for i, d in enumerate(draws):
        for n in d:
            p[i, n - 1] = 1.0
    return p


def compute_features_at_t(presence, t):
    feats = []

    for w in FEAT_WINDOWS:
        s = max(0, t - w)
        feats.append(presence[s:t].sum(axis=0) / max(t - s, 1))

    max_gap = 120
    gaps = np.full(TOTAL_NUMBERS, min(t, max_gap), dtype=np.float32)
    found = np.zeros(TOTAL_NUMBERS, dtype=bool)
    for i in range(t - 1, max(-1, t - max_gap - 1), -1):
        hit = presence[i] > 0
        new_hit = hit & ~found
        gaps[new_hit] = t - 1 - i
        found |= hit
        if found.all():
            break
    feats.append(gaps / max_gap)

    hs = np.zeros(TOTAL_NUMBERS, dtype=np.float32)
    ms = np.zeros(TOTAL_NUMBERS, dtype=np.float32)
    ah = np.ones(TOTAL_NUMBERS, dtype=bool)
    am = np.ones(TOTAL_NUMBERS, dtype=bool)
    for i in range(t - 1, max(-1, t - 35), -1):
        ah &= presence[i] > 0
        am &= presence[i] == 0
        hs[ah] += 1
        ms[am] += 1
        if not ah.any() and not am.any():
            break
    feats.append(hs / 10.0)
    feats.append(ms / 10.0)

    s5 = max(0, t - 5)
    r5 = presence[s5:t]
    if len(r5) > 0:
        w = np.arange(1, len(r5) + 1, dtype=np.float32)
        mom = (r5 * w[:, None]).sum(axis=0) / w.sum()
    else:
        mom = np.zeros(TOTAL_NUMBERS, dtype=np.float32)
    feats.append(mom)

    idx = np.arange(1, TOTAL_NUMBERS + 1, dtype=np.float32)
    feats.append((idx % 2).astype(np.float32))
    feats.append((idx > 40).astype(np.float32))
    feats.append(((idx - 1) // 20) / 3.0)
    feats.append(idx / 80.0)

    return np.stack(feats, axis=1).astype(np.float32)


def build_xy(presence, start_t, end_t):
    min_t = max(start_t, max(FEAT_WINDOWS) + 1)
    n = end_t - min_t
    x = np.empty((n * TOTAL_NUMBERS, 12), dtype=np.float32)
    y = np.empty(n * TOTAL_NUMBERS, dtype=np.float32)
    for j, t in enumerate(range(min_t, end_t)):
        x[j * TOTAL_NUMBERS:(j + 1) * TOTAL_NUMBERS] = compute_features_at_t(presence, t)
        y[j * TOTAL_NUMBERS:(j + 1) * TOTAL_NUMBERS] = presence[t]
    return x, y


def stat_frequency(draws, window=30):
    recent = draws[-window:]
    cnt = np.zeros(TOTAL_NUMBERS, dtype=np.float32)
    for d in recent:
        for n in d:
            cnt[n - 1] += 1
    m = cnt.max()
    return cnt / m if m > 0 else cnt


def stat_gap(draws, window=120):
    recent = draws[-window:]
    last = np.full(TOTAL_NUMBERS, -1, dtype=np.float32)
    for i, d in enumerate(recent):
        for n in d:
            last[n - 1] = i
    total = len(recent)
    gaps = np.where(last >= 0, total - 1 - last, total).astype(np.float32)
    m = gaps.max()
    return gaps / m if m > 0 else gaps


def stat_momentum(draws, window=7):
    recent = draws[-window:]
    sc = np.zeros(TOTAL_NUMBERS, dtype=np.float32)
    for t, d in enumerate(recent):
        w = (t + 1) / max(len(recent), 1)
        for n in d:
            sc[n - 1] += w
    m = sc.max()
    return sc / m if m > 0 else sc


def norm(v):
    m = float(v.max())
    return v / m if m > 0 else v


@dataclass(frozen=True)
class Cfg:
    train_window: int
    rf_trees: int
    rf_depth: int
    et_trees: int
    et_depth: int
    w_ml: float
    w_freq: float
    w_gap: float
    w_mom: float


def run_walk_forward(df, cfg: Cfg, last_n=20, verbose=True):
    issues, draws = load_draws(df)
    presence = presence_matrix(draws)
    eval_start = max(max(FEAT_WINDOWS) + 1, len(draws) - last_n)

    rows = []
    for step, test_idx in enumerate(range(eval_start, len(draws)), start=1):
        start_t = max(0, test_idx - cfg.train_window)
        x_train, y_train = build_xy(presence, start_t, test_idx)
        x_test = compute_features_at_t(presence, test_idx)

        rf = RandomForestClassifier(
            n_estimators=cfg.rf_trees,
            max_depth=cfg.rf_depth,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )
        et = ExtraTreesClassifier(
            n_estimators=cfg.et_trees,
            max_depth=cfg.et_depth,
            random_state=43,
            n_jobs=-1,
            class_weight="balanced",
        )

        rf.fit(x_train, y_train)
        et.fit(x_train, y_train)

        p_rf = rf.predict_proba(x_test)[:, 1]
        p_et = et.predict_proba(x_test)[:, 1]
        p_ml = 0.55 * norm(p_rf) + 0.45 * norm(p_et)

        history = draws[:test_idx]
        p_freq = stat_frequency(history, 30)
        p_gap = stat_gap(history, 120)
        p_mom = stat_momentum(history, 7)

        combined = (
            cfg.w_ml * p_ml + cfg.w_freq * p_freq + cfg.w_gap * p_gap + cfg.w_mom * p_mom
        )

        pred = set(int(i + 1) for i in np.argsort(combined)[::-1][:DRAW_SIZE])
        actual = set(draws[test_idx])
        hits = len(pred & actual)

        rows.append(
            {
                "issue": issues[test_idx],
                "hits": hits,
                "predicted": sorted(pred),
                "actual": sorted(actual),
                "combined": combined,
                "presence": presence,
                "draws": draws,
                "test_idx": test_idx,
            }
        )

        if verbose:
            avg = np.mean([r["hits"] for r in rows])
            print(f"[{step:>2}/{len(range(eval_start, len(draws)))}] Draw {issues[test_idx]}: {hits:>2}/20 hits  avg={avg:.2f}")

    hits_arr = np.array([r["hits"] for r in rows], dtype=np.float32)
    summary = {
        "avg_hits": float(hits_arr.mean()),
        "p_ge_8": float(np.mean(hits_arr >= 8)),
        "p_ge_10": float(np.mean(hits_arr >= 10)),
        "max_hits": int(hits_arr.max()),
        "min_hits": int(hits_arr.min()),
        "rows": rows,
    }
    return summary


def pick_tickets_from_scores(scores, num_tickets=10):
    ranked = np.argsort(scores)[::-1]
    top_pool = ranked[:25]
    weights = scores[top_pool].copy()
    weights = weights / weights.sum()

    tickets = []
    tickets.append(sorted([int(ranked[i] + 1) for i in range(10)]))

    for seed in range(1, num_tickets):
        rng = np.random.RandomState(13 + seed * 17)
        w = weights.copy()
        chosen = []
        for _ in range(10):
            p = w / w.sum()
            idx = rng.choice(len(top_pool), p=p)
            chosen.append(int(top_pool[idx] + 1))
            w[idx] = 0.0
        tickets.append(sorted(chosen))

    return tickets


def build_cover_sets(scores, n_sets=10, set_size=20, pool_size=45):
    """Generate diverse Top20 sets from one score vector."""
    ranked = np.argsort(scores)[::-1]
    pool = ranked[:pool_size]
    base_w = scores[pool].astype(np.float64)
    base_w = base_w / (base_w.sum() + 1e-12)

    sets = []
    sets.append(sorted([int(i + 1) for i in ranked[:set_size]]))

    for s in range(1, n_sets):
        rng = np.random.RandomState(2026 + s * 31)
        w = base_w.copy()

        # Penalize numbers already over-used by previous sets to increase coverage diversity
        usage = np.zeros(len(pool), dtype=np.float64)
        for prev in sets:
            prev_idx = set(x - 1 for x in prev)
            for j, p in enumerate(pool):
                if int(p) in prev_idx:
                    usage[j] += 1.0
        w = w / (1.0 + 0.35 * usage)

        chosen = []
        for _ in range(set_size):
            p = w / (w.sum() + 1e-12)
            j = rng.choice(len(pool), p=p)
            chosen.append(int(pool[j]))
            w[j] = 0.0

        sets.append(sorted([x + 1 for x in chosen]))

    return sets


def evaluate_cover_mode(rows, n_sets=10):
    """Compare single top20 vs best-of-N top20 cover sets."""
    single_hits = []
    cover_best_hits = []

    for r in rows:
        scores = r["combined"]
        actual = set(r["actual"])

        single = set(int(i + 1) for i in np.argsort(scores)[::-1][:20])
        single_hits.append(len(single & actual))

        cover_sets = build_cover_sets(scores, n_sets=n_sets, set_size=20, pool_size=45)
        best = 0
        for st in cover_sets:
            h = len(set(st) & actual)
            if h > best:
                best = h
        cover_best_hits.append(best)

    s = np.array(single_hits, dtype=np.float32)
    c = np.array(cover_best_hits, dtype=np.float32)
    return {
        "single": s,
        "cover": c,
        "single_avg": float(s.mean()),
        "cover_avg": float(c.mean()),
        "single_p10": float(np.mean(s >= 10)),
        "cover_p10": float(np.mean(c >= 10)),
        "single_p8": float(np.mean(s >= 8)),
        "cover_p8": float(np.mean(c >= 8)),
        "single_max": int(s.max()),
        "cover_max": int(c.max()),
    }

In [ ]:
# ===== Baseline Walk-forward =====
base_cfg = Cfg(
    train_window=TRAIN_WINDOW,
    rf_trees=RF_TREES,
    rf_depth=RF_DEPTH,
    et_trees=ET_TREES,
    et_depth=ET_DEPTH,
    w_ml=W_ML,
    w_freq=W_FREQ,
    w_gap=W_GAP,
    w_mom=W_MOM,
)

base = run_walk_forward(df, base_cfg, last_n=LAST_N, verbose=True)

print("=" * 60)
print(f"Average hits  : {base['avg_hits']:.2f} / 20")
print("Random baseline: 5.00 / 20")
print(f"vs Random      : {base['avg_hits'] - 5:+.2f}")
print(f"P(hits>=8)     : {base['p_ge_8']:.2f}")
print(f"P(hits>=10)    : {base['p_ge_10']:.2f}")
print(f"Range          : {base['min_hits']} ~ {base['max_hits']}")
print("=" * 60)

wf_df = pd.DataFrame([{"issue": r["issue"], "hits": r["hits"]} for r in base["rows"]])
display(wf_df)

In [ ]:
# ===== Cover mode evaluation (single top20 vs best-of-10 top20 sets) =====
cover_stats = evaluate_cover_mode(base["rows"], n_sets=10)

print("=" * 60)
print("Cover mode on walk-forward rows")
print(f"Single avg hits     : {cover_stats['single_avg']:.2f}")
print(f"Best-of-10 avg hits : {cover_stats['cover_avg']:.2f}")
print(f"Single P(hits>=8)   : {cover_stats['single_p8']:.2f}")
print(f"Best-of-10 P>=8     : {cover_stats['cover_p8']:.2f}")
print(f"Single P(hits>=10)  : {cover_stats['single_p10']:.2f}")
print(f"Best-of-10 P>=10    : {cover_stats['cover_p10']:.2f}")
print(f"Single max          : {cover_stats['single_max']}")
print(f"Best-of-10 max      : {cover_stats['cover_max']}")
print("=" * 60)

In [ ]:
# ===== Focused Grid Search around your two best seeds =====
# Seed A: Cfg(260,120,8,160,10,0.42,0.30,0.18,0.10)
# Seed B: Cfg(260,180,10,240,12,0.36,0.34,0.20,0.10)

search_space = []

# Neighborhood of Seed A
for tw in [240, 260, 300]:
    for rf_t, et_t in [(120, 160), (160, 220)]:
        search_space.append(Cfg(tw, rf_t, 8, et_t, 10, 0.42, 0.30, 0.18, 0.10))
        search_space.append(Cfg(tw, rf_t, 9, et_t, 11, 0.44, 0.28, 0.18, 0.10))

# Neighborhood of Seed B
for tw in [240, 260, 300]:
    for rf_t, et_t in [(180, 240), (220, 300)]:
        search_space.append(Cfg(tw, rf_t, 10, et_t, 12, 0.36, 0.34, 0.20, 0.10))
        search_space.append(Cfg(tw, rf_t, 11, et_t, 13, 0.34, 0.36, 0.20, 0.10))

# Deduplicate configs
search_space = list(dict.fromkeys(search_space))

records = []
for i, cfg in enumerate(search_space, start=1):
    print(f"[{i}/{len(search_space)}] {cfg}")
    r = run_walk_forward(df, cfg, last_n=LAST_N, verbose=False)
    cv = evaluate_cover_mode(r["rows"], n_sets=10)
    records.append(
        {
            "cfg": cfg,
            "single_avg": r["avg_hits"],
            "single_p_ge_8": r["p_ge_8"],
            "single_p_ge_10": r["p_ge_10"],
            "single_max": r["max_hits"],
            "cover_avg": cv["cover_avg"],
            "cover_p_ge_8": cv["cover_p8"],
            "cover_p_ge_10": cv["cover_p10"],
            "cover_max": cv["cover_max"],
        }
    )

# Priority: tail metrics under cover-mode
ranked = sorted(
    records,
    key=lambda x: (x["cover_p_ge_10"], x["cover_p_ge_8"], x["cover_avg"], x["cover_max"], x["single_avg"]),
    reverse=True,
)

rank_df = pd.DataFrame(
    [
        {
            "cover_p>=10": round(x["cover_p_ge_10"], 3),
            "cover_p>=8": round(x["cover_p_ge_8"], 3),
            "cover_avg": round(x["cover_avg"], 3),
            "cover_max": x["cover_max"],
            "single_avg": round(x["single_avg"], 3),
            "single_p>=10": round(x["single_p_ge_10"], 3),
            "cfg": str(x["cfg"]),
        }
        for x in ranked
    ]
)

print("Top configs (focused search around cfg[1] and cfg[2])")
display(rank_df.head(8))
best_cfg = ranked[0]["cfg"]
print(f"Best cfg = {best_cfg}")

In [ ]:
# ===== Final prediction with best cfg =====

# 如果你不跑 grid search，可手动把 best_cfg 换成 base_cfg
if "best_cfg" not in globals():
    best_cfg = base_cfg

issues, draws = load_draws(df)
presence = presence_matrix(draws)
T = len(draws)
start_t = max(0, T - best_cfg.train_window)

x_train, y_train = build_xy(presence, start_t, T)
x_test = compute_features_at_t(presence, T)

rf = RandomForestClassifier(
    n_estimators=best_cfg.rf_trees,
    max_depth=best_cfg.rf_depth,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
et = ExtraTreesClassifier(
    n_estimators=best_cfg.et_trees,
    max_depth=best_cfg.et_depth,
    random_state=43,
    n_jobs=-1,
    class_weight="balanced",
)

rf.fit(x_train, y_train)
et.fit(x_train, y_train)

p_rf = rf.predict_proba(x_test)[:, 1]
p_et = et.predict_proba(x_test)[:, 1]
p_ml = 0.55 * norm(p_rf) + 0.45 * norm(p_et)
p_freq = stat_frequency(draws, 30)
p_gap = stat_gap(draws, 120)
p_mom = stat_momentum(draws, 7)

final_scores = (
    best_cfg.w_ml * p_ml
    + best_cfg.w_freq * p_freq
    + best_cfg.w_gap * p_gap
    + best_cfg.w_mom * p_mom
)

top20 = sorted([int(i + 1) for i in np.argsort(final_scores)[::-1][:20]])
cover20_sets = build_cover_sets(final_scores, n_sets=10, set_size=20, pool_size=45)
tickets = pick_tickets_from_scores(final_scores, num_tickets=NUM_TICKETS)

print("=" * 60)
print(f"Best cfg: {best_cfg}")
print("Next Draw Top20 (single):")
print(" ".join(f"{n:02d}" for n in top20))
print("=" * 60)
print("Next Draw Top20 Cover Sets (best-of-10 mode):")
for i, st in enumerate(cover20_sets, 1):
    print(f"Set {i:02d}: {' '.join(f'{n:02d}' for n in st)}")

print("=" * 60)
print(f"{NUM_TICKETS} tickets (10 numbers each):")
for i, tk in enumerate(tickets, 1):
    print(f"Ticket {i:02d}: {' '.join(f'{n:02d}' for n in tk)}")

In [ ]:
# ===== Optional: save result to Google Drive =====
# from google.colab import drive
# import json
# from pathlib import Path
#
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/kle_results')
# save_dir.mkdir(parents=True, exist_ok=True)
#
# result = {
#     'best_cfg': str(best_cfg),
#     'top20': top20,
#     'cover20_sets': cover20_sets,
#     'tickets': tickets,
#     'baseline_avg': base['avg_hits'],
#     'cover_stats': cover_stats,
# }
# (save_dir / 'single_notebook_result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
# print(f"Saved to: {save_dir / 'single_notebook_result.json'}")